In [ ]:
import numpy as np
import pandas as pd
import json
from datetime import timedelta
import pycountry
from numpyro import distributions as dist
import arviz as az
from IPython.display import display, Markdown
from matplotlib import pyplot as plt
from matplotlib.pyplot import subplots
from estival.sampling import tools as esamp

from emu_renewal.constants import DATA_PATH, DATE_FORMAT, RUN_DATA_DELAY
from emu_renewal.inputs import get_cont_of_country, get_country_pop
from emu_renewal.renew import MultiStrainModel
from emu_renewal.priors import get_standard_priors
from emu_renewal.run import find_run_start_time, find_run_end_time, get_country_vars, get_alpha_info, get_delta_info, get_mobility_provider, run_calibration
from emu_renewal.targets import SharedDispTarget
from emu_renewal.outputs import run_for_spaghetti, get_spagh_df_from_dict

In [ ]:
mob_source = "fb_singletile_mob"
display(Markdown(f"mobility approach: {mob_source}"))
country = "France"
display(Markdown(f"\n country: {country}"))

In [ ]:
# Build the model
countries = json.load(open(DATA_PATH / f"config/included.json", "r"))
iso3 = pycountry.countries.lookup(country).alpha_3
continent = get_cont_of_country(iso3)
pop = get_country_pop(iso3)
data_start = find_run_start_time(pop, iso3)
end_time = find_run_end_time(iso3, mob_source)
run_start = data_start - timedelta(RUN_DATA_DELAY)
start_str = run_start.strftime(DATE_FORMAT)
end_str = data_start.strftime(DATE_FORMAT)
var_data = get_country_vars(iso3)
delta_var, delta_targ, delta_seed = get_delta_info(iso3, var_data, continent, end_time)
alpha_var, alpha_targ, alpha_seed = get_alpha_info(iso3, var_data, continent, end_time, delta_targ)
start_var = "eu"
var_names = [start_var] + alpha_var
seed_times = [] + alpha_seed
mob_provider = get_mobility_provider(iso3, mob_source)
model = MultiStrainModel(pop, run_start, end_time, var_names, seed_times, mob_provider, False)
times = model.epoch.number_to_datetime(pd.Series(model.model_times))

In [ ]:
priors = get_standard_priors(len(var_names), "weekly_admissions", iso3)
params = {}
for k, v in priors.items():
    params[k] = v if isinstance(v, float) else v.mean
identify_params = {
    "mob_exp": 1.0,
    "beta": 1.75,
}
fixed_params = params | identify_params

In [ ]:
# Get synthetic indicators for calibration
thinning = 7
proc = np.random.normal(0.0, 0.05, 62)
base_params = {"proc": proc} | fixed_params
results = model.renewal_func(**base_params)
indicators = ["weekly_deaths", "weekly_cases"]
outputs = {}
for ind in indicators:
    outputs[ind] = pd.Series(results[ind], index=times)[::thinning]

In [ ]:
fig, axes = plt.subplots(1, len(indicators), figsize=[10, 5])
outputs["weekly_deaths"].plot(ax=axes[0], marker="o", linewidth=0.0, markersize=3.0)
outputs["weekly_cases"].plot(ax=axes[1], marker="o", linewidth=0.0, markersize=3.0)

Rather than calibrating the dispersion parameter, 
in this analysis, the calibration algorithm has been adjusted 
such that the updates to the variable process 
are now assigned uniform distributions over a domain
that spans all the updates used in creating the synthetic data, using the following code:

`params | {"proc": numpyro.sample("proc", dist.Uniform(jnp.repeat(-0.15, self.n_proc_periods), jnp.repeat(0.15, self.n_proc_periods)))}`

In [ ]:
#| output: false
beta_prior = {"beta": dist.Uniform(0.5, 2.5)}
mob_exp_prior = {"mob_exp": dist.Uniform(0.0, 2.0)}
targets = {ind: SharedDispTarget(targ, weight=20) for ind, targ in outputs.items()}
calib, mcmc = run_calibration(model, fixed_params | beta_prior | mob_exp_prior, targets, True, 5000)

In [ ]:
idata = az.from_numpyro(mcmc)
n_samples = 50
idata_sampled = az.extract(idata, num_samples=n_samples)
sample_params = esamp.xarray_to_sampleiterator(idata_sampled)
spaghetti = get_spagh_df_from_dict(run_for_spaghetti(calib, sample_params))

In [ ]:
# Plot fit
fig, axes = plt.subplots(1, len(indicators), figsize=[10, 5])
fig.suptitle("calibration fit")
for i, ind in enumerate(indicators):
    ax = axes[i]
    ax.set_title(ind)
    output = outputs[ind]
    spagh_to_plot = spaghetti[ind]
    for i in spagh_to_plot.columns:
        ax.plot(spaghetti.index, spagh_to_plot[i], color="k", linewidth=0.2)
    ax.plot(output.index, output, marker="o", linewidth=0.0, markersize=3.0)
    ax.tick_params("x", rotation=70)

In [ ]:
display(Markdown(r"\newpage"))
display(Markdown(f"beta input: {fixed_params['beta']}"))
display(Markdown(f"mob_exp input: {fixed_params['mob_exp']}"))
display(Markdown("approximately recovers values of key parameters"))
fig = az.plot_trace(idata)
plt.tight_layout()

In [ ]:
display(Markdown(r"\newpage"))
display(Markdown("recovers general trajectory of variable process"))
fig, ax = plt.subplots(1, 1)
fig.suptitle("recovery of variable process")
proc_times = times[::7]
for params in sample_params:
    ax.plot(proc_times, np.cumsum(params["proc"]), color="k", linewidth=0.2)
ax.plot(proc_times, np.cumsum(proc, axis=0), marker="o", linewidth=0.0);

In [ ]:
display(Markdown(r"\newpage"))
display(Markdown("doesn't recover individual updates to variable process"))
fig, axes = subplots(9, 7, figsize=[20, 16], sharex=True)
flat_axes = axes.ravel()
plot_width = 0.15
for i in range(len(proc)):
    ax = flat_axes[i]
    fig = az.plot_posterior(idata.posterior["proc"][:, :, i], ax=ax, hdi_prob="hide", point_estimate=None)
    fig.vlines(proc[i], 0.0, 5.0, color="k")
    ax.set_title("")
    ax.set_ylabel("")
    ax.set_xlabel("")
    ax.set_xlim([-plot_width, plot_width])
flat_axes[-1].remove()
fig = ax.get_figure()
fig.tight_layout()